### Testing Private Network Gateway (PNG) connectivity from serverless

Validates Databricks serverless → private destination connectivity through a Private Network Gateway (PNG).

This notebook attempts a TCP connection from serverless compute to a PostgreSQL instance (scott-griz-postgresql.scott-griz-png.internal.com, port 5432) that is reachable only over the PNG, addressed by its private hostname.  Replace HOST with the FQDN you listed in your PNG destinations

#### Prerequisites
- Run on serverless compute in a workspace whose NCC has the PNG attached
- The target host is listed in the PNG destinations
- A private DNS resolver (reached via the PNG) resolves the private hostname
- Create secrets to store your PostgreSQL user and password for cell 8


In [0]:
%sh
nslookup scott-griz-postgresql.scott-griz-png.internal.com
nc -vz -w 4 scott-griz-postgresql.scott-griz-png.internal.com 5432

The DNS side is working. nslookup returned 64:ff9b:1::100:ac19:84 in my environment, the NAT64 address PNG maps your VM's private IPv4 to, so the gateway, the destination entry and the private DNS zone link are all doing their jobs. The "recursion not available" lines are harmless warnings from the resolver.

Why nc and Python fail anyway
- nslookup talks to the DNS server directly and accepts the AAAA answer.
- nc and Python's getaddrinfo ask for both A and AAAA records. The PNG resolver only gives a useful answer for AAAA, so the A lookup fails and the libc resolver reports Resource temporarily unavailable. That matches the scoket.gaierror from the "test connection cell" 


In [0]:
import socket
# Replace HOST with the FQDN you listed in your PNG destinations and PORT with the port number if not 5432
HOST = "scott-griz-postgresql.scott-griz-png.internal.com"   # FQDN listed in the PNG destinations
PORT = 5432

# PNG only routes IPv6, so resolve with getaddrinfo (not gethostbyname)
infos = socket.getaddrinfo(HOST, PORT, proto=socket.IPPROTO_TCP)
for fam, _, _, _, addr in infos:
    print(fam.name, addr[0])

fam, stype, proto, _, addr = infos[0]
s = socket.socket(fam, stype, proto)
s.settimeout(10)
try:
    s.connect(addr)
    print("TCP connect OK")
except Exception as e:
    print("TCP connect failed:", e)
finally:
    s.close()

If this "test connection Cell" raises socket.gaierror (Python's address-lookup error), it's the same A-record problem as nc — which is why the "PostgreSQL connection cell" asks for IPv6 explicitly.

In [0]:
%pip install psycopg2-binary

In [0]:
dbutils.library.restartPython()

In [0]:

import socket
import psycopg2

# Replace HOST with the FQDN you listed in your PNG destinations and PORT with the port number if not 5432
HOST = "scott-griz-postgresql.scott-griz-png.internal.com"
addr = socket.getaddrinfo(HOST, 5432, socket.AF_INET6, socket.SOCK_STREAM)[0][4][0]

conn = psycopg2.connect(
    host=HOST, hostaddr=addr, port=5432, dbname="my_test_db",
    user=dbutils.secrets.get("key-vault-secret", "pvt-pg-user"),
    password=dbutils.secrets.get("key-vault-secret", "pvt-pg-password"),
    sslmode="prefer", connect_timeout=10,
)
with conn.cursor() as cur:
    cur.execute("SELECT version(), inet_client_addr()")
    print(cur.fetchone())
conn.close()

Your output should be similar to "('PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit', '172.25.0.4')"